# 🕵️ Case file 01 — The Gold bag leak

> **Mission briefing.** It is 09:14 at Skyline Travel and the support queue is on fire. Basic-fare customers are being quoted Gold-fare baggage rules. As the Observability Detective on duty, your job is to open the activity logs and find out where the answer went wrong.

The assistant searches a policy database before answering — in technical terms, retrieval-augmented generation (RAG). Run the cells from top to bottom in a new kernel.

## Case objective
See how the search scope changes the answer: compare a search across all policies with a search limited to the customer's fare.

## What to expect
The unfiltered run retrieves the Gold policy and quotes it to a Basic-fare customer. The filtered run retrieves the Basic policy. Both runs leave an activity log: `activity-1-rag-no-filter` and `activity-1-rag-filter`.

## Bad practice
Search all policies and use the closest match, without checking whether the policy applies to the customer's fare.

## Good practice
Limit the search to policies for the customer's fare before choosing the closest match.

## Why the good practice works
The assistant cannot tell which fare the customer has unless the search is limited. Without the limit, a policy from a different fare can win the ranking and be quoted as if it applied.

## Deliverable
The two activity-log links and a short comparison of the answers and the policy each run found.


In [1]:
from experiment_utils import close_case, run_rag, show_rag_results, start_quiz, submit_quiz


## 🔮 Your prediction

The customer is on a **Basic** fare and asks how many checked bags they get.

**Which policy do you expect the unfiltered search to return?** And what will the assistant tell the customer?

Write your guess in a comment in the next cell, or double-click this cell and type it below. Commit to it before you run the experiment.

---
*Your guess:*


In [2]:
customer = "How many checked bags do I get on my booking?"
runs = [run_rag(customer, None, "no-filter"), run_rag(customer, "basic", "filter")]
show_rag_results(runs)



all policies
  retrieved: baggage-gold (gold fare) — Gold fare baggage
  query terms matched: 2
  answer: You get **one 23 kg checked bag** included with the Gold fare. A second checked bag costs **60 EUR**.
  tokens: 72 in / 97 out · latency: 126.6s
  activity log: http://localhost:3000/project/langfuse-observability/traces/0ceb971e1d6c74c004039ff863c03bb9

Basic fare only
  retrieved: baggage-basic (basic fare) — Basic fare baggage
  query terms matched: 2
  answer: You don’t get any checked bags included with the Basic fare. You can add a 23 kg checked bag for 45 EUR.
  tokens: 82 in / 103 out · latency: 2.7s
  activity log: http://localhost:3000/project/langfuse-observability/traces/e73cba6b50d3a5979a2a2cc7264122ee


## 🔍 Reading the evidence

Open the unfiltered activity log and inspect the retrieval step. It shows the query went out with no fare filter, and the Gold document won the ranking. Then open the filtered log: the retrieval step records a `basic` filter and only Basic documents were searched.

Both answers sound confident. The difference is in the evidence the assistant was allowed to see — and that is exactly what the activity log makes visible.


## 🏁 Wrap-up

Run the final cell to close the case, show the activity-log links, and flush the traces to Langfuse. Submit the links and your findings.


In [3]:
close_case("activity-1", runs)



case activity-1 closed — session activity-1-681b07b1
all policies: http://localhost:3000/project/langfuse-observability/traces/0ceb971e1d6c74c004039ff863c03bb9
Basic fare only: http://localhost:3000/project/langfuse-observability/traces/e73cba6b50d3a5979a2a2cc7264122ee
activity logs flushed to Langfuse


## 🎁 Bonus — What else would you try?

- Add a topic filter (baggage, cancellation, seating) so unrelated policies never compete.
- Run the same query for a Silver-fare customer and check which policy wins.
- Look at the retrieval step's score in the trace: would a score threshold have caught this leak?


## 🧭 Before you answer — open the retrieval step

Open the unfiltered activity log from the wrap-up cell and click the **`activity-1-retrieval`** step. Read its `input.filter` value and the `output.doc_id` it returned. Then open the filtered log and compare the same fields.

Answer the optional check only after you have seen both retrieval steps.


## 🎯 Optional final check

One question, four options. Correct = 100 points; wrong = 0. The four cases total at most 400.

Run the next cell to see the question, then set `answer` to **1, 2, 3, or 4** in the cell after it and run that cell once. This is secondary: the trace inspection above is the actual exercise.

The checked-in output shows the prompt because verification left `answer = 0`; set your pick and rerun the last cell to score it. Correct options live in `experiment_utils.py`, so the notebook does not spoil them.


In [4]:
start_quiz("activity-1")


optional final check — activity-1
In the filtered run, what filter does the retrieval step record?
  1. No filter — every policy was searched
  2. tier=basic
  3. tier=gold
  4. topic=baggage
set answer to 1, 2, 3, or 4 in the next cell and run it once.


In [5]:
answer = 0  # replace 0 with your choice: 1, 2, 3, or 4
score = submit_quiz("activity-1", answer)


no answer selected yet — set answer to 1, 2, 3, or 4 and run this cell again.
